# ShambaGuard â€” hyperparameter sweeps on Colab

**Before you start**
1. On your laptop run `python scripts/pack_colab.py` and upload `dist/shambaguard_colab.zip` to Google Drive at **MyDrive/ShambaGuard/**.
2. In Colab: *Runtime â†’ Change runtime type â†’ CPU* (no GPU needed; the simulator is the bottleneck).
3. Run the cells top to bottom. Set `ALGOS` in step 5.

**How it behaves**
- Logs and models are written straight to Drive (`MyDrive/ShambaGuard/logs`, `.../models`), so a disconnect loses at most the runs in progress.
- The sweep is resumable: re-run the notebook and finished runs are skipped.
- To go faster, open this notebook in a second tab (*File â†’ Open notebook*, same file) and give it a different `ALGOS`. Each tab is its own machine with 2 CPU cores.
- Free Colab disconnects after ~90 min without interaction and after ~12 h in total. Keep the tab open.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ShambaGuard'
ZIP = f'{DRIVE_DIR}/shambaguard_colab.zip'
WORK = '/content/Shem'
assert os.path.exists(ZIP), f'Upload shambaguard_colab.zip to {DRIVE_DIR} first'
print('found', ZIP, f'({os.path.getsize(ZIP) / 1e6:.1f} MB)')

## 2. Unpack the code and point logs/models at Drive

In [ ]:
import shutil, zipfile
shutil.rmtree(WORK, ignore_errors=True)
zipfile.ZipFile(ZIP).extractall(WORK)

# Results live on Drive; the code copy in /content is throwaway.
for sub in ['logs', 'models/pg', 'models/dqn']:
    target = f'{DRIVE_DIR}/{sub}'
    os.makedirs(target, exist_ok=True)
    link = f'{WORK}/{sub}'
    os.makedirs(os.path.dirname(link), exist_ok=True)
    shutil.rmtree(link, ignore_errors=True)
    os.symlink(target, link)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))
print('CPU cores:', os.cpu_count())

## 3. Install the same package versions as the laptop
torch 2.15 (CPU build), torchrl 0.15.0, tensordict 0.15.6, gymnasium 1.4.0. Takes 1â€“2 minutes. The training scripts run as separate processes, so no runtime restart is needed.

In [ ]:
!pip install -q torch==2.15.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q torchrl==0.15.0 tensordict==0.15.6 gymnasium==1.4.0 pyyaml pytest
!python -c "import sys, torch, torchrl, tensordict, gymnasium; print(sys.version.split()[0], 'torch', torch.__version__, 'torchrl', torchrl.__version__, 'tensordict', tensordict.__version__, 'gymnasium', gymnasium.__version__)"

## 4. Check the environment (tests + a 2-minute smoke run)

In [ ]:
!python -m pytest -q tests
!python training/pg_training.py --algo ppo --run 1 --tag _colab_smoke --total-frames 4096 --set eval_every=100000 final_eval_episodes=2
# remove the smoke run so it does not show up in the summary
import glob
for f in glob.glob('models/pg/ppo_run01_colab_smoke*') + glob.glob('logs/ppo/run01_colab_smoke*'):
    os.remove(f)

## 5. Run the sweep

Each algorithm has 11 runs (0 = from scratch, 1 = warm-start baseline, 2â€“9 = one change each, 10 = best combination; edit run 10 in `configs/<algo>.yaml` after 1â€“9). At 200k frames a run takes roughly 10â€“20 min here, so with 2 workers one algorithm takes about 1.5â€“2 h.

Suggested split across tabs: **tab 1** `['ppo', 'a2c']`, **tab 2** `['dqn', 'reinforce']`.

In [ ]:
ALGOS = ['ppo']          # any of 'ppo', 'a2c', 'reinforce', 'dqn'
RUNS = None              # None = all runs in the YAML; or e.g. [0, 1, 2]
WORKERS = 2              # about one per CPU core

runs_arg = '' if RUNS is None else '--runs ' + ' '.join(map(str, RUNS))
!python scripts/run_sweep.py --algos {' '.join(ALGOS)} {runs_arg} --workers {WORKERS}

## 6. Extra seeds for the best run (after the sweep)
The spec asks for each algorithm's best run repeated with 3 seeds. Seed 0 is the sweep itself; this adds seeds 1 and 2 as `runNN_seed1`, `runNN_seed2`.

In [ ]:
BEST = {'ppo': 1}        # algo -> best run number, e.g. {'ppo': 3, 'dqn': 10}
for algo, run in BEST.items():
    !python scripts/run_sweep.py --algos {algo} --runs {run} --seeds 1 2 --workers {WORKERS}

## 7. Results table (also saved to Drive as logs/sweep_summary.csv)

In [ ]:
!python scripts/summarize_sweeps.py --baselines

## 8. Watch a run in progress (optional)
Per-run console output is in `logs/console/<algo>_<run_id>.out`.

In [ ]:
!ls -t logs/console | head
!tail -n 3 logs/console/$(ls -t logs/console | head -1)